# DSSATLab tutorial: two crop simulations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdelrahmanAmr3/dssatlab/blob/master/notebook/dssatlab_tutorial.ipynb)

Run the cells from top to bottom. You will run the DSSAT crop model twice:

1. **Case 1**: DSSAT's own example, exactly as it ships. Nothing to edit.
2. **Case 2**: your own crop. You fill in three small files (weather, soil, management) and run it.

You do not need to know Python. Every cell is one or two lines.

## Set up

In [ ]:
%pip install -q "dssatlab[plot,yaml] @ git+https://github.com/AbdelrahmanAmr3/dssatlab.git@v0.8.0"

The next cell builds the DSSAT program on Colab. The first time it takes several minutes; after that it takes a moment.

In [ ]:
import dssatlab as dl
dssat = dl.install()

## Case 1: DSSAT's own example

A maize experiment from Gainesville, Florida, with 6 treatments (rainfed or irrigated, low or high nitrogen). The next cell downloads DSSAT's own files for it into a folder called `case1_gainesville`.

In [ ]:
!wget -q -P case1_gainesville https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Maize/UFGA8201.MZX https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Weather/UFGA8201.WTH https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop/Soil/SOIL.SOL https://raw.githubusercontent.com/DSSAT/dssat-csm-os/develop/Data/Genotype/MZCER048.CUL https://raw.githubusercontent.com/DSSAT/dssat-csm-os/develop/Data/Genotype/MZCER048.ECO https://raw.githubusercontent.com/DSSAT/dssat-csm-os/develop/Data/Genotype/MZCER048.SPE
!ls case1_gainesville

Run it. `dl.run` runs every treatment in the file.

In [ ]:
result1 = dl.run("case1_gainesville/UFGA8201.MZX")

Look at the results, one row per treatment:

- `HWAM`: grain yield (kg/ha)
- `CWAM`: total above-ground biomass (kg/ha)
- `MDAT`: date the crop matured
- `IRCM`: irrigation water applied (mm)

In [ ]:
import pandas as pd
pd.DataFrame(result1.summary())[["TNAM", "PDAT", "MDAT", "HWAM", "CWAM", "IRCM"]]

Leaf area index over the season, for each treatment:

In [ ]:
result1.plot("LAID");

## Case 2: your own crop

Now the same maize experiment file, but with **your** weather, soil and management. You only edit three files:

| File | What it holds |
|---|---|
| `my_weather.csv` | one row per day: temperature, rain, solar radiation |
| `my_soil.csv` | one row per soil layer |
| `my_experiment.yaml` | sowing, irrigation, fertilizer, cultivar, starting soil, simulation settings |

Two values must stay as they are, because the experiment file fixes them: the weather `station` code (`UFGA`) and the soil profile `soil_id` (`IBMZ910014`). Everything else is yours to change.

This example is summer maize at Kafr El-Sheikh, Egypt, in 2020. The next cell copies the experiment file into a new folder `case2_my_data` and downloads the example weather (real daily data from NASA POWER).

In [ ]:
!mkdir -p case2_my_data
!cp case1_gainesville/UFGA8201.MZX case1_gainesville/MZCER048.* case2_my_data/
!wget -q -P case2_my_data https://raw.githubusercontent.com/AbdelrahmanAmr3/dssatlab/master/notebook/tutorial_data/my_weather.csv
!ls case2_my_data

### Your weather

Your own weather goes in a CSV file with exactly these columns. Units are DSSAT's own: nothing is converted for you. If your data is in Excel, rename your columns to these names, use dates like `2020-05-15`, and save as CSV.

| Column | Meaning |
|---|---|
| `station` | station code, must be `UFGA` here |
| `latitude`, `longitude` | degrees |
| `elevation` | metres |
| `date` | `YYYY-MM-DD`, one row for every day, no gaps |
| `srad` | solar radiation, MJ/m2/day |
| `tmax`, `tmin` | daily maximum and minimum temperature, °C |
| `rain` | rainfall, mm |

In [ ]:
pd.read_csv("case2_my_data/my_weather.csv").head()

### Your soil

One row per soil layer. Edit the numbers and run the cell to save your changes.

In [ ]:
%%writefile case2_my_data/my_soil.csv
soil_id,salb,slro,sldr,slpf,slb,slll,sdul,ssat,srgf,sbdm,sloc
IBMZ910014,0.12,75,0.3,1.0,10,0.20,0.36,0.48,1.0,1.25,1.4
IBMZ910014,0.12,75,0.3,1.0,30,0.21,0.37,0.48,0.7,1.30,1.0
IBMZ910014,0.12,75,0.3,1.0,60,0.22,0.38,0.47,0.4,1.35,0.7
IBMZ910014,0.12,75,0.3,1.0,80,0.23,0.38,0.47,0.2,1.40,0.5
IBMZ910014,0.12,75,0.3,1.0,100,0.23,0.38,0.47,0.1,1.40,0.4

### Your management

Sowing, irrigation, fertilizer, cultivar and the starting soil. Edit anything and run the cell to save your changes.

In [ ]:
%%writefile case2_my_data/my_experiment.yaml
# My maize crop: Kafr El-Sheikh, Egypt, summer 2020 (treatment 1).
# Every date is in quotes: "YYYY-MM-DD". Leave a section out to keep what the FileX has.

treatments:
  1:
    planting:
      date: "2020-05-15"          # sowing date
      method: "S"                 # S = seed
      distribution: "R"           # R = rows
      population: 7.0             # plants per m2
      row_spacing: 70.0           # cm
      depth: 5.0                  # cm

    irrigation:                   # one block per irrigation: date, mm of water, method
      - {date: "2020-05-16", amount: 60, method: "IR001"}
      - {date: "2020-05-26", amount: 60, method: "IR001"}
      - {date: "2020-06-05", amount: 60, method: "IR001"}
      - {date: "2020-06-15", amount: 60, method: "IR001"}
      - {date: "2020-06-25", amount: 60, method: "IR001"}
      - {date: "2020-07-05", amount: 60, method: "IR001"}
      - {date: "2020-07-15", amount: 60, method: "IR001"}
      - {date: "2020-07-25", amount: 60, method: "IR001"}
      - {date: "2020-08-04", amount: 60, method: "IR001"}
      - {date: "2020-08-14", amount: 60, method: "IR001"}

    fertilizer:                   # n = kg of nitrogen per hectare; FE005 = urea, AP001 = broadcast
      - {date: "2020-05-15", material: "FE005", application: "AP001", depth: 5, n: 120}
      - {date: "2020-06-09", material: "FE005", application: "AP001", depth: 5, n: 120}

    cultivar:
      crop: "MZ"
      code: "IB0035"              # a cultivar code that exists in MZCER048.CUL

    initial_conditions:           # the soil on the start date, layer by layer
      date: "2020-05-01"
      previous_crop: "WH"         # wheat before maize
      residue_mass: 1500          # kg/ha of wheat residue
      layers:
        - {depth: 10,  water: 0.30, nh4: 1.0, no3: 6.0}
        - {depth: 30,  water: 0.32, nh4: 0.8, no3: 4.0}
        - {depth: 60,  water: 0.33, nh4: 0.5, no3: 3.0}
        - {depth: 100, water: 0.34, nh4: 0.4, no3: 2.0}

    controls:
      start_date: "2020-05-01"    # the simulation starts here; weather must cover it
      water: "Y"                  # simulate water
      nitrogen: "Y"               # simulate nitrogen
      output_interval: 1          # write daily output

### Check and run

`check()` reads all your files and lists every problem at once. It changes nothing and does not start DSSAT.

In [ ]:
sim = dl.Simulation(
    "case2_my_data/UFGA8201.MZX",
    treatment=1,
    weather="case2_my_data/my_weather.csv",
    soil="case2_my_data/my_soil.csv",
    management="case2_my_data/my_experiment.yaml",
)
problems = sim.check()

When every line says OK, run the simulation:

In [ ]:
result2 = sim.run()
pd.DataFrame(result2.summary())[["PDAT", "MDAT", "HWAM", "CWAM", "IRCM"]]

In [ ]:
result2.plot("LAID");

### When something is wrong

Here is a soil file whose `soil_id` does not match the experiment file. `check()` tells you what is wrong and how to fix it, before anything runs.

In [ ]:
%%writefile case2_my_data/bad_soil.csv
soil_id,salb,slro,sldr,slpf,slb,slll,sdul,ssat,srgf,sbdm,sloc
IBMZ910099,0.12,75,0.3,1.0,10,0.20,0.36,0.48,1.0,1.25,1.4
IBMZ910099,0.12,75,0.3,1.0,30,0.21,0.37,0.48,0.7,1.30,1.0
IBMZ910099,0.12,75,0.3,1.0,60,0.22,0.38,0.47,0.4,1.35,0.7
IBMZ910099,0.12,75,0.3,1.0,80,0.23,0.38,0.47,0.2,1.40,0.5
IBMZ910099,0.12,75,0.3,1.0,100,0.23,0.38,0.47,0.1,1.40,0.4

In [ ]:
dl.Simulation(
    "case2_my_data/UFGA8201.MZX",
    treatment=1,
    weather="case2_my_data/my_weather.csv",
    soil="case2_my_data/bad_soil.csv",
    management="case2_my_data/my_experiment.yaml",
).check()

The fix is to use the right `soil_id` (`IBMZ910014`), as in `my_soil.csv` above.

## Your turn

Change the numbers in the soil and management cells above and run again, for example irrigate less, sow later or add nitrogen, and compare `HWAM`. To use your own weather, upload your CSV to `case2_my_data` in the Colab file panel and use its name in the `weather=` line.

## Case 3: compare with what you measured

Use Case 1's `result1` to compare observed data with the Gainesville maize results. It holds all six treatments of `UFGA8201.MZX`; we only compare treatments 1 and 2 here, under scenario `base`.


**All numbers and dates below are invented for the tutorial, not real measurements.** Replace them with your own observed data when comparing your crop. `HWAM` is grain yield (kg[dm]/ha), `ADAT` is anthesis date, `MDAT` is maturity date, and `LAID` is leaf area index (m2/m2). Nothing is converted.

Leave `date` blank for Summary values. Fill it in for Plant growth on that day. Use `yyyy-mm-dd` for every date. The LAID dates below fall inside Case 1's 1982 season, from planting on February 26 to maturity around June 30.

In [ ]:
%%writefile case1_gainesville/my_observed.csv
# Invented observed data for the tutorial, not real measurements.
# HWAM: kg[dm]/ha; LAID: m2/m2; ADAT/MDAT and date: yyyy-mm-dd.
scenario,treatment,date,HWAM,ADAT,MDAT,LAID
base,1,,4500,1982-05-10,1982-06-28,
base,2,,6000,1982-05-12,1982-06-30,
base,1,1982-04-15,,,,1.8
base,1,1982-05-15,,,,3.2
base,2,1982-05-15,,,,4.0


Compare the observed data with the run. Each pair shows the observed value, simulated value and their difference (simulated minus observed). Dates in the table use DSSAT's `yyyyddd` integers; date errors are in days.

In [ ]:
ev = dl.evaluate(result1, "case1_gainesville/my_observed.csv")
ev.to_dataframe()

Look at RMSE, mean bias and Willmott's d-index for each variable. These statistics need at least two pairs; a variable with one pair still has its individual error.

In [ ]:
ev.statistics

Compare grain yield on a plot. Points on the dashed 1:1 line match exactly; points above it mean the simulated yield is higher.

In [ ]:
dl.plot_evaluation(ev, variable="HWAM");